# Resultados del experimento HA2: escenarios A, B, C y G

**MISW4202 · Grupo 12, Solventa**

## Resumen

Se procesaron las 27.533 peticiones registradas en la Fase 5 (7 escenarios × 3 corridas) y se calcularon las métricas del ASR HA2 para los escenarios **A** (baseline), **B** (lentitud), **C** (caída total) y **G** (concurrencia).

**Las dos metas se cumplen en los cuatro escenarios.** La disponibilidad fue del 100 % en todos, sin una sola petición fallida en 18.217. En B y C, donde el proveedor se degrada y el circuito llega a abrirse, el 100 % de las conmutaciones ocurrió por debajo de 1 s; la más lenta tardó 6,4 ms.

| | A · Baseline | B · Lentitud | C · Caída total | G · Concurrencia |
|---|---|---|---|---|
| Peticiones | 2.337 | 5.449 | 5.789 | 4.642 |
| Disponibilidad | 100 % | 100 % | 100 % | 100 % |
| Tasa de errores | 0 % | 0 % | 0 % | 0 % |
| Conmutaciones < 1 s | N/A | 100 % | 100 % | N/A |
| Cache hit rate | N/A | 100 % | 100 % | N/A |
| Latencia p50 / p95 | 498 / 647 ms | 0,7 / 75 ms | 0,7 / 1,7 ms | 3.779 / 4.006 ms |

Dos advertencias para leer bien la tabla, y las dos importan:

- **A y G no ejercitan el mecanismo.** Los dos corrieron con el proveedor sano, así que no hubo conmutaciones ni consultas a caché. Sus celdas de conmutación y hit rate salen `N/A` porque el denominador es cero, no porque falte el dato.
- **La latencia de G no mide el circuit breaker.** Los ~4 s son la saturación del proveedor simulado con 50 usuarios en paralelo. En G el circuito nunca se abrió.

## 1. Datos cargados

Para el análisis se cargaron tres fuentes, cada una con un papel distinto:

**`resultados/adaptador.jsonl`** es la fuente principal. Es el registro que escribe el Adaptador, con una fila por petición y las columnas que definimos en el esquema de medición: estado del circuito al inicio y al final, `hit_miss`, `tiempo_conmutacion_ms`, `latencia_total_ms` y el resultado del journey. De aquí salen todas las métricas del ASR: 27.533 peticiones repartidas en 7 escenarios y 3 corridas cada uno.

**Los `results_stats.csv` de Locust** aportan el throughput y la latencia vista desde el cliente. Son agregados por endpoint (dos filas por corrida con conteos y percentiles), así que sirven para medir cuántas peticiones por segundo sostuvo el sistema y para contrastar la latencia externa con la que mide el Adaptador, pero no para las métricas del ASR. Les faltan el estado del circuito, `hit_miss` y `tiempo_conmutacion_ms`, y hay un límite de fondo: para Locust una respuesta del proveedor y una servida desde caché son el mismo HTTP 200, así que no distingue un journey exitoso de uno degradado.

**Los `manifest.json`** registran las condiciones con las que corrió cada escenario: modo del proveedor simulado, usuarios, duración y los parámetros del breaker.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

# La raiz del repositorio se localiza buscando la carpeta de datos, para no
# depender de cuantos niveles separan al cuaderno de la raiz.
RAIZ = next(
    padre for padre in [Path.cwd(), *Path.cwd().parents] if (padre / "resultados").is_dir()
)
sys.path.insert(0, str(RAIZ))

from analisis.procesamiento.carga import (
    ESCENARIOS_ENTREGABLE,
    cargar_manifiestos,
    cargar_peticiones,
)
from analisis.procesamiento import metricas

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

SALIDAS = RAIZ / "analisis" / "procesamiento" / "salidas"

carga = cargar_peticiones()
df = carga.peticiones
print(carga.resumen_calidad())

No se descartó ninguna fila. Las dos incidencias que reporta la carga son ajustes necesarios sobre lo que emite la instrumentación:

- Las 9.562 peticiones servidas por el proveedor llegan con `hit_miss = "N/A"`. Nunca consultaron la caché, así que se dejan fuera del denominador del hit rate; contarlas como miss lo hundiría de forma artificial.
- El `ejecucion_id` (`run1_rep1..3`) se repite en los siete escenarios, de modo que agrupar solo por él mezclaría corridas distintas. La llave es siempre el par `(escenario, ejecucion_id)`.

### Condiciones de cada corrida

Antes de mirar cualquier número conviene revisar qué se probó: el modo del proveedor simulado decide si el mecanismo llegó a activarse.

In [ ]:
condiciones = cargar_manifiestos()
condiciones[condiciones.escenario.isin(ESCENARIOS_ENTREGABLE)].drop_duplicates(
    subset=["escenario", "modo_mock", "usuarios", "duracion_s"]
)[["escenario", "modo_mock", "usuarios", "duracion_s", "timeout_ms", "fail_max", "reset_timeout_s", "ttl_s"]]

Aquí queda claro por qué A y G se comportan igual: los dos usan `modo=normal`. G se diferencia solo en la carga (50 usuarios durante 120 s frente a 10 durante 60 s), así que mide concurrencia con el proveedor sano, no degradación bajo carga.

## 2. Totales, disponibilidad, errores y hit rate

Estas son las fórmulas que definimos como equipo al diseñar el experimento:

- Disponibilidad = (exitosos + degradados) / total × 100
- Tasa de errores = fallidos / total × 100
- Conmutaciones < 1 s = las que tardaron menos de 1 s ÷ **las que efectivamente conmutaron** × 100
- Cache hit rate = HIT / (HIT + MISS) × 100

In [ ]:
resumen = metricas.tabla_resumen(df)
columnas = [
    "escenario", "total_peticiones", "exitosos", "degradados", "fallidos",
    "disponibilidad_pct", "tasa_errores_pct", "peticiones_conmutadas",
    "pct_conmutaciones_bajo_1s", "cache_hit_rate_pct",
    "latencia_p50_ms", "latencia_p95_ms", "latencia_p99_ms", "latencia_max_ms",
]
resumen[resumen.escenario.isin(ESCENARIOS_ENTREGABLE)][columnas].round(2)

La columna `degradados` muestra dónde actuó el mecanismo: 5.295 peticiones en B y 5.643 en C se respondieron desde Redis en vez de fallar. Esas son las que sostienen el 100 % de disponibilidad, y todas encontraron perfil en caché (hit rate del 100 %).

Sobre el denominador del porcentaje bajo 1 s: son las peticiones que conmutaron, no el total. En A y G no conmutó ninguna, así que la métrica queda indefinida y se reporta `N/A`. Poner 0 % sugeriría que el mecanismo falló y poner 100 % que funcionó; ninguna de las dos cosas pasó, sencillamente no se ejercitó.

## 3. Distribución de latencia

In [ ]:
display(Image(filename=str(SALIDAS / "distribucion_latencia.png")))

Cada caja abarca del percentil 25 al 75 y la línea naranja es la mediana; el eje es logarítmico porque los escenarios difieren en varios órdenes de magnitud.

**A** se sitúa cerca de los 500 ms: es el costo normal de consultar Open Finance. **B y C caen por debajo de 1 ms**, unas 700 veces más rápido, porque con el circuito abierto casi todas las peticiones se resuelven contra Redis sin salir a la red. Que C quede aún más comprimido que B tiene sentido: en caída total el proveedor rechaza al instante, mientras que en lentitud algunas peticiones alcanzan a esperar el timeout.

La caja de **G** está en otra escala, sobre los 3.800 ms. Insistimos en el punto porque es fácil malinterpretarlo: no es el mecanismo siendo lento, es el proveedor simulado saturado con 50 usuarios concurrentes. En G el circuito permaneció cerrado todo el tiempo.

In [ ]:
display(Image(filename=str(SALIDAS / "latencia_en_el_tiempo.png")))

Cada punto es una petición a lo largo de una corrida. En **A** y **G** la nube es plana de principio a fin, como corresponde a un proveedor que no cambia de comportamiento.

En **B** y **C** se ve el quiebre: durante los primeros ~10 s las peticiones cuestan unos 100 ms contra el proveedor y, cuando este se degrada, la nube se desploma a la banda de 1 ms. Los puntos sueltos que quedan arriba después del quiebre son los reintentos periódicos del breaker, que vuelve a probar el proveedor cada cierto tiempo.

## 4. Estado del circuito en el tiempo

In [ ]:
display(Image(filename=str(SALIDAS / "estado_circuito_y_latencia.png")))

Esta es la gráfica que mejor explica el mecanismo. La línea roja escalonada es el estado del circuito (eje derecho) y los puntos azules la latencia (eje izquierdo, logarítmico).

Al segundo 10 el proveedor se degrada y el circuito salta de `CLOSED` a `OPEN`. En ese mismo instante la latencia cae dos órdenes de magnitud, de ~100 ms a ~0,7 ms: es el efecto de dejar de esperar al proveedor y responder desde Redis.

Las líneas verticales naranjas marcan los reintentos. Aparecen cada ~10 s porque `reset_timeout` está en 10 s: el breaker deja pasar una petición de prueba, el proveedor sigue caído y el circuito se reabre. Ese patrón regular es exactamente lo que se espera con `fail_max=1`, y es también la razón de los puntos altos aislados de la gráfica anterior.

B acumula 47 disparos en la corrida frente a 6 de C. La diferencia viene del modo de fallo: en lentitud cada reintento gasta el timeout completo antes de darse por vencido, mientras que en caída total el rechazo es inmediato.

## 5. Cumplimiento de las metas

In [ ]:
display(Image(filename=str(SALIDAS / "disponibilidad_por_escenario.png")))
display(Image(filename=str(SALIDAS / "conmutaciones_bajo_1s.png")))

Las cuatro barras de disponibilidad tocan el 100 % y quedan sobre la línea de meta del 99,9 %. No hubo ni un fallo en 18.217 peticiones.

En la segunda gráfica B y C llegan al 100 %, con máximos de 6,4 ms y 5,3 ms. El margen contra el objetivo de 1 s es enorme: unas 156 veces. A y G aparecen con `N/A` porque no tuvieron conmutaciones que medir.

Vale la pena separar dos cosas que suelen confundirse. Conmutar es barato, cuestión de milisegundos; lo caro es **detectar** que hay que hacerlo, y eso cuesta los 700 ms del timeout. Aun así, la petición más lenta de B llegó a 712,8 ms sumando detección y conmutación, todavía por debajo del segundo.

## 6. Throughput bajo concurrencia (escenario G)

In [ ]:
display(Image(filename=str(SALIDAS / "throughput_escenario_g.png")))

Este es el único dato que aportan los CSV de Locust. Los usuarios suben a 50 en los primeros segundos (línea gris) y el throughput se estabiliza en ~12 peticiones por segundo, sin caídas ni degradación progresiva durante los 120 s. Las tres corridas quedan prácticamente superpuestas.

El aporte de G, entonces, es mostrar que **la carga por sí sola no dispara el breaker**: 4.642 peticiones concurrentes sin una sola conmutación espuria. El circuito solo reacciona a degradación real del proveedor, que es el comportamiento deseado.

## 7. Consistencia entre corridas

In [ ]:
repro = metricas.reproducibilidad(df)
repro[repro.escenario.isin(ESCENARIOS_ENTREGABLE)][
    ["escenario", "corridas", "disponibilidad_pct_std",
     "pct_conmutaciones_bajo_1s_std", "latencia_p50_ms_min", "latencia_p50_ms_max"]
].round(3)

Desviación estándar cero en disponibilidad y en el porcentaje bajo 1 s: las tres corridas de cada escenario dieron lo mismo. Las medianas de latencia también se mueven poco (en B, entre 0,716 y 0,733 ms). Los resultados no dependen de la corrida.

## Conclusiones

El ASR HA2 se cumple en los escenarios de este bloque. Con el proveedor degradado (lento en B, caído en C) el sistema mantuvo el 100 % de disponibilidad y conmutó a caché en milisegundos, muy lejos del límite de 1 s.

Tres cosas que conviene declarar al consolidar la evidencia:

1. **Falta el cruce entre degradación y concurrencia.** G corrió con el proveedor sano, así que ningún escenario probó qué pasa cuando el proveedor se degrada y hay carga alta al mismo tiempo.
2. **La disponibilidad del 100 % supone la caché poblada.** En B y C todas las peticiones derivadas encontraron perfil. El escenario F cubre el caso contrario.
3. **El registro interno tiene ~5 % más peticiones que Locust** (en A, 773 contra 708), porque el Adaptador registra peticiones fuera de la ventana de medición de Locust. No afecta a las métricas, que son proporciones, pero explica que los conteos no cuadren si se comparan las dos fuentes.

El análisis del costo diferenciado entre la petición que dispara el corte y las que encuentran el circuito ya abierto se aborda por separado; el motor de métricas ya expone ese desglose mediante `metricas.por_poblacion()`.